<a href="https://colab.research.google.com/github/saisathvik485/BIS-LAB-5M-/blob/main/graph_path_optimization_using_genetic_algorithm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import random
import copy

# --- 1. PROBLEM SETUP (GRAPH DEFINITION) ---
# Adjacency matrix representing distance between 5 graph nodes (0 to 4)
# float('inf') can be used to simulate missing edges / unidirectional limits
GRAPH = [
    [0, 10, 15, 20, 25],
    [10, 0, 35, 25, 30],
    [15, 35, 0, 30, 5],
    [20, 25, 30, 0, 15],
    [25, 30, 5, 15, 0]
]
NUM_NODES = len(GRAPH)

# --- 2. GA HYPERPARAMETERS ---
POP_SIZE = 30
GENERATIONS = 100
CROSSOVER_RATE = 0.8
MUTATION_RATE = 0.2

# --- 3. HELPER FUNCTIONS ---
def calculate_path_distance(path):
    """Fitness Function: Computes total path length. Lower values are better."""
    total_distance = 0
    for i in range(len(path) - 1):
        dist = GRAPH[path[i]][path[i+1]]
        if dist == float('inf'):
            return float('inf') # Invalid path penalty
        total_distance += dist

    # Corrected Loop Closure: Add return route distance back to the starting node [0]
    total_distance += GRAPH[path[-1]][path[0]]
    return total_distance

def create_individual():
    """Generates a random valid node path permutation sequence."""
    individual = list(range(NUM_NODES))
    random.shuffle(individual)
    return individual

def selection(population, fitnesses):
    """Tournament Selection: Pick the best individual from a random group sample."""
    tournament_size = 3
    selected_idx = random.sample(range(len(population)), tournament_size)
    best_idx = min(selected_idx, key=lambda idx: fitnesses[idx])
    return copy.deepcopy(population[best_idx])

def ordered_crossover(parent1, parent2):
    """Ordered Crossover (OX): Prevents duplicate node allocations in permutations."""
    if random.random() > CROSSOVER_RATE:
        return copy.deepcopy(parent1), copy.deepcopy(parent2)

    size = len(parent1)
    child1, child2 = [-1] * size, [-1] * size

    # Select random segment bounds
    start, end = sorted(random.sample(range(size), 2))

    # Copy baseline slice to children
    child1[start:end+1] = parent1[start:end+1]
    child2[start:end+1] = parent2[start:end+1]

    # Fill remaining gaps sequentially with elements from the alternate parent
    def fill_child(child, parent):
        p_idx = 0
        for c_idx in range(size):
            if child[c_idx] == -1:
                while parent[p_idx] in child:
                    p_idx += 1
                child[c_idx] = parent[p_idx]

    fill_child(child1, parent2)
    fill_child(child2, parent1)

    return child1, child2

def mutation(individual):
    """Inversion Mutation: Reverses an isolated path segment sequence."""
    if random.random() < MUTATION_RATE:
        start, end = sorted(random.sample(range(len(individual)), 2))
        # Corrected: Explicitly cast the reversed iterator to a list to prevent structural breakdown
        individual[start:end+1] = list(reversed(individual[start:end+1]))
    return individual

# --- 4. GA ENGINE CORE ---
def run_genetic_algorithm():
    # Initialize initial population matrix
    population = [create_individual() for _ in range(POP_SIZE)]

    best_individual = None
    best_fitness = float('inf')

    for gen in range(GENERATIONS):
        fitnesses = [calculate_path_distance(ind) for ind in population]

        # Track the global elite minimum
        for i, fit in enumerate(fitnesses):
            if fit < best_fitness:
                best_fitness = fit
                best_individual = copy.deepcopy(population[i])

        next_generation = []
        # Elitism step: Automatically preserve the optimal path found so far
        next_generation.append(copy.deepcopy(best_individual))

        while len(next_generation) < POP_SIZE:
            p1 = selection(population, fitnesses)
            p2 = selection(population, fitnesses)

            c1, c2 = ordered_crossover(p1, p2)
            c1 = mutation(c1)
            c2 = mutation(c2)

            next_generation.append(c1)
            if len(next_generation) < POP_SIZE:
                next_generation.append(c2)

        population = next_generation

    return best_individual, best_fitness

# --- 5. EXECUTION ---
if __name__ == "__main__":
    optimal_path, min_distance = run_genetic_algorithm()

    # Format string presentation showing loop return point
    closed_loop = optimal_path + [optimal_path[0]]
    path_visual = " -> ".join(str(node) for node in closed_loop)

    print("\n=== OPTIMAL SOLUTION FOUND ===")
    print(f"Shortest Path Sequence : {path_visual}")
    print(f"Total Minimum Distance  : {min_distance} units")




=== OPTIMAL SOLUTION FOUND ===
Shortest Path Sequence : 1 -> 0 -> 2 -> 4 -> 3 -> 1
Total Minimum Distance  : 70 units
